In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
outputPath = projectPath / 'pipelines' / 'operational' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
gapCases = pl.read_parquet(dataPrepPath / 'opticalGapCases.parquet').filter(pl.col('locfEvaluable'))
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
comparisonRows = predictions.join(gapCases.select('rowId', 'ndvi', 'locfNdvi', 'gapAgeDays', 'locfGapRange'), on='rowId', how='inner').with_columns((pl.col('prediction') - pl.col('ndvi')).abs().alias('radarAbsError'), (pl.col('locfNdvi') - pl.col('ndvi')).abs().alias('locfAbsError')).with_columns((pl.col('radarAbsError') - pl.col('locfAbsError')).alias('errorDifference'), (pl.col('radarAbsError') < pl.col('locfAbsError')).alias('radarWins'))
radarVsLocf = comparisonRows.group_by('model', 'featureSet', 'seed', 'locfGapRange').agg(pl.col('gapAgeDays').min().alias('minimumGapDays'), pl.col('gapAgeDays').max().alias('maximumGapDays'), pl.col('radarAbsError').mean().alias('radarMae'), pl.col('locfAbsError').mean().alias('locfMae'), pl.col('errorDifference').mean().alias('meanErrorDifference'), pl.col('radarWins').mean().alias('radarWinRate'), pl.len().alias('rowCount')).sort('model', 'featureSet', 'seed', 'minimumGapDays')
radarVsLocf.write_parquet(outputPath / 'radarVsLocf.parquet')
assert comparisonRows.select(pl.col('rowId').n_unique()).item() == 754
assert radarVsLocf.select(pl.col('rowCount').sum()).item() == comparisonRows.height
radarVsLocf
